# 🏔️ Quantum Ascent — Basecamp 2: Gates & Circuits

<div style="border-left:5px solid #1f7a4d;background:#eaf6ee;color:#0f3d22;padding:12px 16px;border-radius:4px">
<b>📋 Mission briefing.</b> In Basecamp 1 you spun a single coin. Now you'll learn the <b>moves</b> that steer it — and stack those moves into <b>circuits</b>. You'll meet the bit-flip <b>X</b> and the phase-flip <b>Z</b>, discover that the <i>order</i> you apply gates in changes the result, see why every quantum move can be <i>undone</i>, and dodge the #1 beginner trap in Qiskit: how it numbers qubits.<br><br><i>New symbols and a little matrix or two show up here — but every one is just a tidy name for something you'll first watch happen in the widget. No memorising. We build it up slowly, one move at a time.</i>
</div>

**By the end of this basecamp you can:**
- Recognise the bit-flip **X** and the phase-flip **Z**, and describe **H** as the 'mixer' — all as simple moves of the arrow
- Stack gates into a circuit and predict the result, knowing that **order matters** (gates don't always commute)
- Explain why every gate is **reversible** (unitarity) — a quantum computer never quietly loses information
- Read Qiskit's qubit ordering correctly and step around the **endianness trap** when interpreting measurement bitstrings

*Estimated time: 50 min · Best experienced with the [course website](https://quantum-ascent-77617.web.app) open in another tab.*

In [ ]:
# ✅ Setup — run me first! (works locally and on Google Colab)
import importlib.util, os, sys, subprocess, urllib.request

def _ensure_q2q():
    for rel in (".", "..", "../.."):
        if os.path.isdir(os.path.join(rel, "q2q")):
            sys.path.insert(0, os.path.abspath(rel))
            break
    if importlib.util.find_spec("q2q") is not None:
        return
    # On Colab: install the pinned SDKs and fetch the course helpers
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "qiskit==2.3.1", "qiskit-aer==0.17.2", "pylatexenc"], check=False)
    base = ("https://raw.githubusercontent.com/sahgyan9/quantum-ascent/"
            "main/notebooks/q2q/")
    os.makedirs("q2q", exist_ok=True)
    for fname in ("__init__.py", "checkers.py", "widgets.py", "oracles.py",
                  "latex_macros.py", "targets.py", "quiz.py", "progress.py"):
        urllib.request.urlretrieve(base + fname, os.path.join("q2q", fname))
    sys.path.insert(0, os.path.abspath("."))

_ensure_q2q()
from q2q import checkers, quiz, targets, progress
from q2q.widgets import show_widget
print("Setup complete — you're ready to climb. 🏔️")

## 2.1 Gates are moves you can stack

Quick recap from Basecamp 1. You already met **two gates** without us making a fuss about it:

- the **Hadamard** ($H$) — it set a resting coin *spinning* (tipped $|0\rangle$ down to the fair-coin equator);
- the **rotation** $R_y(\theta)$ — the θ *slider* in gate form, tilting the arrow to any bias you like.

That's the whole idea of a **gate**: a *move* that takes the qubit's arrow from where it is to somewhere new. And just like dance steps, moves can be **stacked** — do one, then the next, then the next. A stack of gates is a **circuit**.

This basecamp is about two brand-new moves and what happens when you combine them. Let's meet them in the playground *before* we name any math.

<div style="border-left:5px solid #0e7490;background:#eaf4f6;color:#0c363f;padding:12px 16px;border-radius:4px">
<b>🔭 Exercise 1.</b> Open the <b>Gate Playground</b> below. It's a single qubit and a row of gate buttons — click one to drop it on the wire and watch the arrow move. <br>1️⃣ Click <b>X</b> once (starting from |0⟩). Where does the arrow go? Read the explanation — this move has a plain-English nickname. <br>2️⃣ Hit <b>Reset</b>, then click <b>H</b> (fair spin), then click <b>Z</b>. Did the P(0)/P(1) bars change at all? Look very closely. <br>3️⃣ Something <i>did</i> change even though the odds didn't — the widget tells you what. Keep that mystery in mind; it's the seed of Basecamp 3.
</div>

In [ ]:
show_widget("gate-playground")

### Naming the two new moves

Time to rename what you just watched with the words the textbooks use.

**The bit-flip $X$.** Clicking $X$ swapped $|0\rangle$ and $|1\rangle$ — a definite 0 became a definite 1. It's the quantum version of the classical NOT gate: it flips the bit. On the Bloch arrow it's a half-turn (180°) that swings the north pole to the south pole.

**The phase-flip $Z$.** This one is sneakier. Clicking $Z$ *did not move the probabilities at all* — a fair coin stayed a fair coin. What it changed was the **phase**: the *sign* hiding on the $|1\rangle$ amplitude. It turns $|1\rangle$ into $-|1\rangle$, leaving $|0\rangle$ alone.

> 🧊 **Why should I care about an invisible sign?** Great question — hold it. A phase you can't see in a single measurement is exactly the ingredient that lets amplitudes *cancel* later (interference). That cancellation is where quantum computers get their edge. Basecamp 3 cashes this in; for now, just notice that $Z$ changes the *state* without changing the *odds*.

*(For the curious — each gate is also a little 2×2 matrix. Feel free to skim; we'll use them properly, not memorise them.)*

$$X = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}, \qquad Z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}, \qquad H = \tfrac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}$$

<div style="border-left:5px solid #0e7490;background:#eaf4f6;color:#0c363f;padding:12px 16px;border-radius:4px">
<b>🔭 Exercise 2.</b> Back to the playground — this time test whether <b>order matters</b>. <br>1️⃣ <b>Reset</b>, then build <b>H</b> then <b>Z</b>. Note where the arrow points. <br>2️⃣ <b>Reset</b>, then build <b>Z</b> then <b>H</b> (same two gates, swapped). Does the arrow land in the <i>same</i> place? <br>3️⃣ Predict before you peek — then decide the quick check below.
</div>

In [ ]:
# Quick check — does the order of gates matter? (Predict first!)
quiz.ask("m2-order-matters")

You just discovered that **gates usually don't commute**: $H$ then $Z$ is a *different* circuit from $Z$ then $H$. Order is not a detail — it's part of the recipe. (Sound familiar? "Socks then shoes" and "shoes then socks" use the same two actions and give very different results.)

Keep that in your pocket. Now let's build these circuits for real, in code.

## 2.2 Building circuits in Qiskit

Same virtual lab as Basecamp 1 — `QuantumCircuit` to build, `AerSimulator` to run. We'll also grab **NumPy** this time, purely so we can talk to gates as little matrices when we want to. Run the setup:

In [ ]:
# Our tools again — the same two as Basecamp 1, plus NumPy for matrices.
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

### Worked example: a gate and its undo

Here's a circuit that applies $H$ **twice** to one qubit. Predict it before you run: $H$ sets the coin spinning… so what does a *second* $H$ do? (You saw the answer in the playground — two H's snapped the arrow right back.)

Instead of measuring, we'll check the circuit's **overall action** directly with `check_unitary_equiv`, which compares what your circuit *does* to a target — here, the "do-nothing" identity matrix `np.eye(2)`. A green PASS means the two H's truly cancel:

In [ ]:
# Build H then H on one qubit.
qc_hh = QuantumCircuit(1)
qc_hh.h(0)
qc_hh.h(0)

# Does this circuit do... nothing at all? Compare it to the identity (do-nothing) gate.
checkers.check_unitary_equiv(qc_hh, np.eye(2))

That green light *is* **unitarity**: every quantum gate is **reversible**. Each move has an undo, so information is never destroyed along the way — a quantum computer only ever *rearranges* amplitudes, never erases them. ($H$ happens to be its own undo; so are $X$ and $Z$.)

In [ ]:
# Quick check — what do two H's in a row do?
quiz.ask("m2-unitary-undo")

<div style="border-left:5px solid #b45309;background:#fdf3e7;color:#5c2d0c;padding:12px 16px;border-radius:4px">
<b>⛏️ Task 1.</b> Now stack three moves and discover a hidden identity. Build <code>qc_flip</code> by applying, in order, <b>H</b>, then <b>Z</b>, then <b>H</b> to a single qubit — exactly the H-then-Z you played with, capped by one more H. <br><i>Predict first:</i> Z alone does nothing to the odds… but sandwiched between two Hadamards? The checker compares your circuit to a mystery gate — run it and see which famous gate H·Z·H really is.
</div>

In [ ]:
# Build H, then Z, then H on one qubit — then discover what single gate it equals.
qc_flip = QuantumCircuit(1)
# YOUR CODE HERE

# Nothing to change below — we compare your circuit's action to the mystery target.
checkers.check_unitary_equiv(qc_flip, targets.M2_FLIP)

#### 🔬 Analysis

Surprise — **H·Z·H = X**, the bit-flip! A phase-flip, wrapped in
two Hadamards, becomes an honest bit-flip. Here's the intuition: the first $H$
rotates your point of view so that "phase" and "bit value" swap roles; $Z$ does
its phase-flip in that rotated frame; the last $H$ rotates back. What looked
invisible ($Z$ changing only a sign) became fully visible ($X$ flipping 0↔1).
This "sandwich a gate between Hadamards to change what it does" trick shows up
everywhere in quantum computing — and it's your first taste of *why* those
hidden phases are worth caring about.

## 2.3 Two qubits, and the trap that catches everyone

Real circuits have more than one qubit. The moment you add a second, Qiskit springs a famous surprise on newcomers — so let's walk into it on purpose, with eyes open.

Make a **2-qubit** circuit. The qubits are numbered **qubit 0** and **qubit 1**. When you measure, Qiskit hands back a little string like `'10'` — and here's the trap:

> ⚠️ **Qiskit writes qubit 0 on the *right*.** The string is read **right-to-left**: the rightmost character is qubit 0, the next one left is qubit 1, and so on. (Computer scientists call this *little-endian* — the "littlest" qubit sits at the right end, exactly like the ones digit sits at the right end of the number 42.)

So a result of `'10'` means **qubit 1 = 1, qubit 0 = 0**. Let's prove it: we'll flip **qubit 1** and watch which character lights up.

In [ ]:
# Worked example: a 2-qubit circuit. Flip qubit 1, then measure everything.
qc_demo = QuantumCircuit(2)
qc_demo.x(1)                 # flip qubit 1 only
qc_demo.measure_all()

# Run it and read the bitstring. Predict first: which character turns into a 1?
counts_demo = checkers.run_and_tally(qc_demo, shots=200)

# Qubit 1 is the LEFT character, so flipping it should give '10' every time.
checkers.check_counts_close(counts_demo, {"10": 1.0})

See it? We touched qubit **1** and the **left** character became `1` → `'10'`. Now you predict the mirror image before you build it:

In [ ]:
# Quick check — flip qubit 0 instead. Which bitstring? (Predict, then build it below.)
quiz.ask("m2-endianness")

<div style="border-left:5px solid #b45309;background:#fdf3e7;color:#5c2d0c;padding:12px 16px;border-radius:4px">
<b>⛏️ Task 2.</b> Build <code>qc_endian</code>: a <b>2-qubit</b> circuit that flips <b>only qubit 0</b> and then measures. The measurement is already wired for you — you just add the one flip. <br><i>Predict first:</i> qubit 0 is the <b>rightmost</b> character, so which bitstring should show up on every shot? Get it right and you've beaten the trap that catches almost every beginner.
</div>

In [ ]:
# A 2-qubit circuit. Flip ONLY qubit 0 — predict the bitstring before you run!
qc_endian = QuantumCircuit(2)
# YOUR CODE HERE

# Nothing to change below — the measurement, then 512 shots checked against '01'.
qc_endian.measure_all()
counts = checkers.run_and_tally(qc_endian, shots=512)
checkers.check_counts_close(counts, {"01": 1.0})

#### 🔬 Analysis

Flipping qubit **0** lit up the **right** character → `'01'`,
the mirror image of the worked example. That's the whole trap: **Qiskit reads
right-to-left.** Whenever a result looks backwards, you now know the culprit —
count the qubits from the right. Two habits save you every time: read bitstrings
right-to-left, and when in doubt, flip one known qubit (like we just did) to see
where it lands.

<div style="border:1px dashed #1f7a4d;background:#eaf6ee;color:#0f3d22;padding:12px 16px;border-radius:8px">
<b>🎨 Make it yours — analogy time.</b> Everyone's brain hooks onto different things. Copy the prompt below into <i>your</i> favorite AI (ChatGPT, Claude, Gemini) and get <b>quantum gates, circuit order, and reversibility</b> explained through <i>your</i> world. The precise definition is baked into the prompt, so the AI can't drift into pop-science myths. Want more control? Use the <a href="https://quantum-ascent-77617.web.app/analogy-studio.html">Analogy Studio</a>.

<pre style="white-space:pre-wrap;background:#ffffff;border:1px solid #cfe6d8;color:#0f3d22;border-radius:6px;padding:10px">I'm learning quantum computing. Explain quantum gates and circuits using an analogy from MY background: [YOUR HOBBY/FIELD HERE].

Ground rules — your analogy MUST respect these facts:
1) A gate is a reversible move applied to a qubit's state; a circuit is a sequence of such moves read left to right.
2) X is the bit-flip (|0⟩↔|1⟩). Z is the phase-flip: it multiplies |1⟩ by -1, changing the state but NOT the measurement probabilities.
3) Gates generally do NOT commute — the order matters (H then Z differs from Z then H).
4) Every gate is unitary, i.e. reversible: applying it has an undo, so no information is lost (H·H returns you to the start).

End by telling me where the analogy breaks down.</pre>
</div>

## 🎓 Log your climb — claim your completion code

You stacked gates into circuits, discovered that **H·Z·H = X**, and stepped
around Qiskit's endianness trap. Let's bank it! Run the cell below: it re-checks
your **Task 1** (`qc_flip`) and **Task 2** (`qc_endian`) right here in this
kernel and — if they pass — prints a personal **completion code**.

Copy that code into the **“Log your notebook”** box on the
[Basecamp 2 page](https://quantum-ascent-77617.web.app/module.html?id=02#claim)
to light up this camp on your Ascent map and bank your climber XP. 🏔️

In [ ]:
progress.claim_basecamp_2(qc_flip, qc_endian)

<div style="border-left:5px solid #1f7a4d;background:#eaf6ee;color:#0f3d22;padding:12px 16px;border-radius:4px">
<b>🚩 Basecamp 2 reached!</b> You turned single gates into circuits: you met the bit-flip $X$ and the phase-flip $Z$, proved that order matters, saw that every gate is reversible, discovered the H·Z·H = X sandwich, and learned to read Qiskit's bitstrings right-to-left. You now speak the grammar of quantum circuits.
</div>

**Next steps:**
1. 🧠 Take the [Basecamp 2 quiz](https://quantum-ascent-77617.web.app/module.html?id=02#quiz) on the website to earn your XP and badge.
2. 🧗 Continue the ascent: **Basecamp 3: Entanglement — where two qubits share a single, spooky fate**.

---
*Stuck on a task? Compare with the worked solutions: [`solutions/02_gates_and_circuits_solutions.ipynb`](solutions/02_gates_and_circuits_solutions.ipynb). Try honestly first — the struggle is where the learning happens.*